In [ ]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax
import jax.numpy as jnp
import optax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_mouse import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import trainer
from models import bom
from models import hgm


# jax.config.update("jax_enable_x64", True)

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(trainer)
importlib.reload(bom)
importlib.reload(hgm)

In [ ]:
manual_hyperparameters = {
	# 'max_epochs': 4000,
	'max_epochs': 4000,
	'max_patience': 4000 / 1,
	'learning_rate': 1E-0,
	'dimensions': 2,
	'ccadiag_lambda': 0.6,
	'ccaunit_lambda': 1.0 - 0.6,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'wb_reg_lambda': 1E-1,
	'wm_reg_lambda': 1E-9,
}

# Functions

In [ ]:
class Lambdas(NamedTuple):
	loss_lambdas: tuple[float, float]
	reg_lambdas: tuple[float, float]
	cca_lambdas: tuple[float, float]

In [ ]:
def train_model(model, splits, N, B, S, hyperparams=None):
	#region hyperparamters ===============================================================
	seeds = jax.random.PRNGKey(0), jax.random.PRNGKey(1), jax.random.PRNGKey(2), jax.random.PRNGKey(3), jax.random.PRNGKey(4)

	global manual_hyperparameters
	hyperparameters = manual_hyperparameters.copy()
	if hyperparams is not None:
		for key in hyperparams:
			hyperparameters[key] = hyperparams[key]

	max_epochs = hyperparameters['max_epochs']
	max_patience = hyperparameters['max_patience']

	learning_rate = hyperparameters['learning_rate']

	n_dimensions = hyperparameters['dimensions']

	lambdas = Lambdas(
		loss_lambdas=(hyperparameters['loss_man_lambda'], hyperparameters['loss_res_lambda']),
		reg_lambdas=(hyperparameters['wb_reg_lambda'], hyperparameters['wm_reg_lambda']),
		cca_lambdas=(hyperparameters['ccadiag_lambda'], hyperparameters['ccaunit_lambda'])
	)
	#endregion ===========================================================================

	n_neurons = N.shape[1]
	n_classes = S.shape[1]
	n_behaviors = B.shape[1]

	N, B, S = jnp.asarray(N), jnp.asarray(B), jnp.asarray(S)

	# training loop ==============================
	params_per_split = list()
	(losses_train_per_split,
	 man_losses_train_per_split,
	 res_losses_train_per_split,
	 mpe_losses_train_per_split) = list(), list(), list(), list()
	(losses_per_split,
	 man_losses_per_split,
	 res_losses_per_split,
	 mpe_losses_per_split) = list(), list(), list(), list()
	for s_id, (train, eval) in enumerate(splits):
		#region splitting and initializing (this also moves the data to the GPU) ==========================
		N_train, N_eval = N[train], N[eval]
		B_train, B_eval = B[train], B[eval]
		S_train, S_eval = S[train], S[eval]

		params = model.init_params(N_train, B_train, S_train,
								   seeds,
								   (n_neurons, n_dimensions),
								   (n_classes, n_dimensions),
								   (n_behaviors, n_neurons),
								   * ((n_dimensions, n_neurons),) if model.get_type() else (),
								  )

		optimizer = optax.adam(learning_rate=optax.cosine_decay_schedule(init_value=learning_rate, decay_steps=max_epochs, alpha=1E-4))
		# optimizer = optax.adam(learning_rate=learning_rate)
		optimizer = optax.masked(optimizer, {'prep': False, 'model': True})
		opt_state = optimizer.init(params)
		#endregion ========================================================================================

		# train the split
		(params,
		 losses_train, man_losses_train, res_losses_train, mpe_losses_train,
		 losses, man_losses, res_losses, mpe_losses, epoch_end
		 ) = trainer.train(lambdas,
						   max_epochs, max_patience,
						   model, optimizer.update,
						   opt_state, params,
						   N_train, B_train, S_train,
						   N_eval,  B_eval,  S_eval,)

		# save the split
		params_per_split.append(jax.device_get(params))

		losses_train_per_split.append(jax.device_get(losses_train))
		man_losses_train_per_split.append(jax.device_get(man_losses_train))
		res_losses_train_per_split.append(jax.device_get(res_losses_train))
		mpe_losses_train_per_split.append(jax.device_get(mpe_losses_train))

		losses_per_split.append(jax.device_get(losses))
		man_losses_per_split.append(jax.device_get(man_losses))
		res_losses_per_split.append(jax.device_get(res_losses))
		mpe_losses_per_split.append(jax.device_get(mpe_losses))

		# log
		print(f'{f'Stopped after {epoch_end} epochs: ' if epoch_end < max_epochs else ''}Split {s_id} completed. ', end='')

	# ============================================

	# return
	return (params_per_split,
			np.stack(losses_train_per_split),
			np.stack(man_losses_train_per_split),
			np.stack(res_losses_train_per_split),
			np.stack(mpe_losses_train_per_split),
			np.stack(losses_per_split),
			np.stack(man_losses_per_split),
			np.stack(res_losses_per_split),
			np.stack(mpe_losses_per_split))

In [ ]:
def train_models(model, filenames, classes, behaviors, traces, hyperparams=None, save_name=None):
	models_params = list()
	models_losses, models_man_losses, models_res_losses, models_mpe_losses = list(), list(), list(), list()
	models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train = list(), list(), list(), list()

	for i in range(len(filenames)):
		print(f'\n{i+1}/{len(filenames)}: {filenames[i]}... ', end='')

		splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces[i].shape[0]), classes[i][['run']].values))

		(params_per_split,
		 losses_train_per_split, man_losses_train_per_split, res_losses_train_per_split, mpe_losses_train_per_split,
		 losses_per_split, man_losses_per_split, res_losses_per_split, mpe_losses_per_split
		 ) = train_model(model, splits,
						 traces[i].values,
						 behaviors[i].values,
						 classes[i].values,
						 hyperparams)

		models_params.append(params_per_split)

		models_losses_train.append(losses_train_per_split)
		models_man_losses_train.append(man_losses_train_per_split)
		models_res_losses_train.append(res_losses_train_per_split)
		models_mpe_losses_train.append(mpe_losses_train_per_split)

		models_losses.append(losses_per_split)
		models_man_losses.append(man_losses_per_split)
		models_res_losses.append(res_losses_per_split)
		models_mpe_losses.append(mpe_losses_per_split)

	if save_name is not None:
		train_utils.save_params(filenames, models_params, save_name)
		train_utils.save_losses(filenames, [
			np.array([models_losses_train[i], models_man_losses_train[i], models_res_losses_train[i], models_mpe_losses_train[i],
					  models_losses[i], models_man_losses[i], models_res_losses[i], models_mpe_losses[i]]) for i in range(len(filenames))
		], save_name)

	return (models_params,
			models_losses_train, models_man_losses_train, models_res_losses_train, models_mpe_losses_train,
			models_losses, models_man_losses, models_res_losses, models_mpe_losses)

# read & prep

In [ ]:
ll = f'll{int(manual_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams_old.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	'wb_reg_lambda': 1E-1,
	# 'wb_reg_lambda': swept_hyperparameters[ll]['bom']['means']['wb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	'wb_reg_lambda': manual_hyperparameters['wb_reg_lambda'],
	# 'wb_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wb_reg_lambda']['geometric'],
	'wm_reg_lambda': manual_hyperparameters['wm_reg_lambda'],
	# 'wm_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wm_reg_lambda']['geometric'],
}

In [ ]:
# file_ids = list(range(0, 8+1))
file_ids = [2]	# paper
plot_id = 2

In [ ]:
# read
path = os.path.join(read_utils.get_data_path(), '6163622/sessions')
(file_ids, filenames,
 traces,
 continuous_behaviors,
 motion_pcs,
 motion_masks) = read_utils.get_datas(path, file_ids,
									  with_whisking=True)

# clear nan values
for i in range(len(file_ids)):
	# mask = motion_pcs[i][0].notna().all(axis=1)
	mask = continuous_behaviors[i].notna().all(axis=1)

	traces[i] = traces[i][mask]
	motion_pcs[i][0] = motion_pcs[i][0][mask]
	motion_pcs[i][1] = motion_pcs[i][1][mask]
	continuous_behaviors[i] = continuous_behaviors[i][mask]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty or type(label) is not str else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep

def reconstruct_calcium(spikes, fs, decay=2.0):
	dt = 1.0 / fs
	gamma = np.exp(-dt / decay)
	trace = np.zeros(spikes.shape, dtype=np.float32)
	for t in range(1, spikes.shape[0]):
		trace[t, :] = gamma * trace[t-1, :] + spikes[t, :]
	return trace

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(reconstruct_calcium(trace.values, fs=sr), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

behaviors = [mpc[0] for mpc in motion_pcs]

classes = [
	pd.DataFrame(cb[['running_speed']].values > np.quantile(cb[['running_speed']].values, 0.05), columns=['run'])
	for cb in continuous_behaviors
]

In [ ]:
limit_n = 5
plot_idx = file_ids.index(plot_id)

# GCAMP

In [ ]:
plot_utils.plot_traces(traces[plot_idx].iloc[:, :limit_n],
					   classes[plot_idx],
					   label='running',
					   title=f'[{filenames[plot_idx]}] traces', ylimits=None)

### BOM

In [ ]:
(boms_params,
 boms_losses_train, boms_man_losses_train, boms_res_losses_train, boms_mpe_losses_train,
 boms_losses_test, boms_man_losses_test, boms_res_losses_test, boms_mpe_losses_test,
 ) = train_models(bom,
				  filenames,
				  classes,
				  behaviors,
				  traces,
				  hyperparams=bom_reg_lambdas,
				  save_name=os.path.join('fly-2023-schaffer', 'new', ll, f'ns_bom_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}'))  #save_name=None

### HGM

In [ ]:
(hgms_params,
 hgms_losses_train, hgms_man_losses_train, hgms_res_losses_train, hgms_mpe_losses_train,
 hgms_losses_test, hgms_man_losses_test, hgms_res_losses_test, hgms_mpe_losses_test,
 ) = train_models(hgm,
				  filenames,
				  classes,
				  behaviors,
				  traces,
				  hyperparams=hgm_reg_lambdas,
				  save_name=os.path.join('fly-2023-schaffer', 'new', ll, f'ns_hgm_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}'))  #save_name=None

### losses

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_train, boms_losses_train,
							  title='hgm vs bom: train',
							  ylimits=0.10)

In [ ]:
# plot_utils.plot_losses_splits(filenames,
# 							  hgms_man_losses_train, boms_man_losses_train,
# 							  title='hgm vs bom: man train',
# 							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_res_losses_train, boms_res_losses_train,
							  title='hgm vs bom: res train',
							  ylimits=0.10)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_train, boms_mpe_losses_train,
							  title='hgm vs bom: mpe train',
							  ylimits=0.10)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_test, boms_losses_test,
							  title='hgm vs bom: test',
							  ylimits=0.10)

In [ ]:
# plot_utils.plot_losses_splits(filenames,
# 							  hgms_man_losses_test, boms_man_losses_test,
# 							  title='hgm vs bom: man test',
# 							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_res_losses_test, boms_res_losses_test,
							  title='hgm vs bom: res test',
							  ylimits=0.10)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_test, boms_mpe_losses_test,
							  title='hgm vs bom: mpe test',
							  ylimits=0.10)